In [17]:
import pandas as pd
import random
import os
import unittest

In [18]:
# Reflex Agent Logic (Perception to Action)

def process_email_ticket(row, ticket_counter):
    """
    Reflex Agent rules: reads current email inputs and returns ticket classifications.
    """
    try:
        # Convert text to lowercase for accurate keyword matching
        email = str(row.get('Email_Address', '')).lower()
        content = str(row.get('Email_Content', '')).lower()
        
        # Generate unique Ticket ID
        ticket_id = f"TKT-{ticket_counter:04d}"
    
        # Identify Sender Type based on domain and keywords
        if "student" in email and "@stbrigids.ie" in email:
            sender_type = "Student"
        elif "@stbrigids.ie" in email:
            sender_type = "Staff"
        else:
            sender_type = "External User"

        # Classify Issue Type using keyword detection
        if any(keyword in content for keyword in ["password", "login", "access"]):
            issue_type = "Account Issue"
        elif any(keyword in content for keyword in ["computer", "printer", "screen"]):
            issue_type = "Hardware Issue"
        elif any(keyword in content for keyword in ["wifi", "internet", "network"]):
            issue_type = "Network Issue"
        else:
            issue_type = "General IT Issue"  


        # Detect Priority Level
        if any(keyword in content for keyword in ["urgent", "immediately", "cannot work"]):
            priority = "High"
        else:
            priority = "Normal"
            
        # Assign IT Staff Member randomly (to ensure even workload)
        staff_member = random.choice(["Staff 1", "Staff 2", "Staff 3", "Staff 4"])
        
        
        # Return the new processed fields
        return pd.Series([ticket_id, sender_type, issue_type, priority, staff_member]) 

    except Exception as e:
        # Exception Handling for individual rows: prevents one bad email from crashing the whole system
        print(f"Error processing ticket {ticket_counter}: {e}")
        return pd.Series([f"TKT-{ticket_counter:04d}", "Error", "Error", "Error", "Unassigned"])



In [19]:
# Main Execution Flow

def run_triage_system():
    filename = "emails.csv"
    
    # 1. Exception Handling for File Loading (Input)
    try:
        print(f"Loading dataset: {filename}...")
        df = pd.read_csv(filename)
    except FileNotFoundError:
        print(f"Error: '{filename}' not found. Please run the dataset generation cell first.")
        return # Stop execution safely
    except pd.errors.EmptyDataError:
        print(f"Error: '{filename}' is empty. No data to process.")
        return
    except Exception as e:
        print(f"Unexpected error loading dataset: {e}")
        return

    # 2. Exception Handling for Data Processing (Logic)
    try:
        print("Applying Reflex Agent rules...")
        new_columns = df.apply(lambda row: process_email_ticket(row, row.name + 1001), axis=1)
        new_columns.columns = ['Ticket_ID', 'Sender_Type', 'Issue_Type', 'Priority_Level', 'Assigned_Staff']
        
        
        processed_df = pd.concat([df, new_columns], axis=1)
    except Exception as e:
        print(f"Error during Agent processing: {e}")
        return
    
    # 3. Exception Handling for File Saving (Output)
    issue_categories = ["Account Issue", "Hardware Issue", "Network Issue", "General IT Issue"]
    
    print("\nGenerating output files...")
    for issue in issue_categories:
        try:
            # Filter data for the specific issue category
            category_df = processed_df[processed_df['Issue_Type'] == issue]
            
            # Create a clean file name (e.g., Account_Issue_Tickets.csv)
            output_filename = f"{issue.replace(' ', '_')}_Tickets.csv"
            
            # Save to CSV
            category_df.to_csv(output_filename, index=False)
            print(f" -> Saved {len(category_df)} tickets to '{output_filename}'")
        except PermissionError:
            # Common error if the user has the CSV open in Excel while running the code
            print(f"Error: Permission denied. Please close '{output_filename}' if it is open in another program.")
        except Exception as e:
            print(f"Error saving '{output_filename}': {e}")
            
    print("\nAutomated Triage Process Complete.")


In [20]:

def test_agent_logic(email, content):
    email = str(email).lower()
    content = str(content).lower()
    
    if "student" in email and "@stbrigids.ie" in email: sender = "Student"
    elif "@stbrigids.ie" in email: sender = "Staff"
    else: sender = "External User"

    if any(k in content for k in ["password", "login", "access"]): issue = "Account Issue"
    elif any(k in content for k in ["computer", "printer", "screen"]): issue = "Hardware Issue"
    elif any(k in content for k in ["wifi", "internet", "network"]): issue = "Network Issue"
    else: issue = "General IT Issue"  

    if any(k in content for k in ["urgent", "immediately", "cannot work"]): priority = "High"
    else: priority = "Normal"
        
    return sender, issue, priority


class TestReflexAgentSafe(unittest.TestCase):
    def test_student_account(self):
        s, i, p = test_agent_logic('student1@stbrigids.ie', 'forgot password')
        self.assertEqual(s, 'Student')
        self.assertEqual(i, 'Account Issue')
        self.assertEqual(p, 'Normal')
        
    def test_staff_urgent(self):
        s, i, p = test_agent_logic('lecturer@stbrigids.ie', 'URGENT printer broken')
        self.assertEqual(s, 'Staff')
        self.assertEqual(i, 'Hardware Issue')
        self.assertEqual(p, 'High')


In [28]:
# Execute the program
if __name__ == "__main__":
    
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.003s

OK


In [29]:
run_triage_system()

Loading dataset: emails.csv...
Applying Reflex Agent rules...

Generating output files...
 -> Saved 14 tickets to 'Account_Issue_Tickets.csv'
 -> Saved 11 tickets to 'Hardware_Issue_Tickets.csv'
 -> Saved 18 tickets to 'Network_Issue_Tickets.csv'
 -> Saved 7 tickets to 'General_IT_Issue_Tickets.csv'

Automated Triage Process Complete.
